# 04_lakebase
Captured 2026-11-21 21:08 UTC

**Operational write path with Part 11-style controls.** Planners propose, entitled approvers decide with a rationale, no one approves their own proposal, every change lands in an append-only audit log, decisions cannot be deleted, and Postgres RLS mirrors the UC region rules (an EU planner cannot write or see US actions).


In [1]:
-- connected as a service principal role without BYPASSRLS
SELECT current_user, rolbypassrls

[["7739855e-67af-424e-a487-b03d30433d3a", false]]

In [2]:
-- synced tables from UC [app.user=maggie.barrett@databricks.com]
SELECT (SELECT count(*) FROM ops.disruption_impact) AS impact_rows, (SELECT count(*) FROM ops.user_entitlement) AS entitlements

[[38, 2]]

In [3]:
-- planner proposes for EU [app.user=7739855e-67af-424e-a487-b03d30433d3a]
INSERT INTO ops.mitigation_action (disruption_id, sku_id, region, action_type, description, proposed_by, proposed_source) SELECT disruption_id, sku_id, region, 'REALLOCATE_INVENTORY', 'Evidence test: move 4 weeks of EU stock', '7739855e-67af-424e-a487-b03d30433d3a', 'PLANNER' FROM ops.disruption_impact WHERE region = 'EU' ORDER BY supply_gap_days DESC LIMIT 1 RETURNING action_id, region, status

[[1, "EU", "PROPOSED"]]

In [4]:
-- planner tries to propose for US (RLS WITH CHECK) [app.user=7739855e-67af-424e-a487-b03d30433d3a]
INSERT INTO ops.mitigation_action (disruption_id, region, action_type, description, proposed_by, proposed_source) VALUES ('test', 'US', 'ALLOCATE_DEMAND', 'should be blocked', '7739855e-67af-424e-a487-b03d30433d3a', 'PLANNER')

ERROR (expected): new row violates row-level security policy for table "mitigation_action"

In [5]:
-- planner sees only EU actions [app.user=7739855e-67af-424e-a487-b03d30433d3a]
SELECT region, count(*) FROM ops.mitigation_action GROUP BY region

[["EU", 1]]

In [6]:
-- planner tries to approve (not entitled) [app.user=7739855e-67af-424e-a487-b03d30433d3a]
UPDATE ops.mitigation_action SET status='APPROVED', decision_rationale='x' WHERE status='PROPOSED' AND region='EU'

ERROR (expected): 7739855e-67af-424e-a487-b03d30433d3a is not entitled to approve mitigations

In [7]:
-- approval without rationale [app.user=maggie.barrett@databricks.com]
UPDATE ops.mitigation_action SET status='APPROVED' WHERE status='PROPOSED' AND region='EU'

ERROR (expected): decision rationale is required

In [8]:
-- exec approves with rationale [app.user=maggie.barrett@databricks.com]
UPDATE ops.mitigation_action SET status='APPROVED', decision_rationale='Stock exists in EU-DE1; approved for 4 weeks' WHERE action_id = (SELECT max(action_id) FROM ops.mitigation_action WHERE status='PROPOSED' AND region='EU') RETURNING action_id, status, decided_by

[[1, "APPROVED", "maggie.barrett@databricks.com"]]

In [9]:
-- audit trail is append-only [app.user=maggie.barrett@databricks.com]
DELETE FROM ops.audit_log

ERROR (expected): permission denied for table audit_log

In [10]:
-- decisions cannot be deleted [app.user=maggie.barrett@databricks.com]
DELETE FROM ops.mitigation_action

ERROR (expected): permission denied for table mitigation_action

In [11]:
-- audit trail [app.user=maggie.barrett@databricks.com]
SELECT action_id, event, actor, at FROM ops.audit_log ORDER BY audit_id DESC LIMIT 6

[[1, "UPDATE:APPROVED", "maggie.barrett@databricks.com", "2026-11-21 21:08:43.212887+00:00"], [1, "INSERT:PROPOSED", "7739855e-67af-424e-a487-b03d30433d3a", "2026-11-21 21:08:42.615224+00:00"]]